<a href="https://colab.research.google.com/github/v-yunusic/REPO-PYTHON-VY/blob/main/transacciones-surtidor-2022/bencinera_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CONTEXTO

El conjunto pertenece a datos registrados por nodos IoT durante los mes de enero a marzo 2022.

Se trata de una serie temporal y contiene en general la siguiente información:

  * Nro de la transacción.
  * Fecha y hora de inicio y de fin de la transacción.
  * Cantidad de litros de combustible despachado.
  * Volumen real registrado por otros nodos IoT en los tanques durante el inicio y fin de la descarga.
  * Estación, surtidor, empresa de origen.
  * Tipo de combustible transaccionado.
  * Geolocalización (si es soportada).
  * Unidad de consumo de destino.
  * Sector de la industria a la que pertenece la estación.

Desde la serie más atómica a la agrupación más general, el orden es el siguiente:
id_bomba --> id_equipo --> id_empresa --> id_industria

# QUEHACERES

* Estandarizar columna nombre_prod, definir clases/tipos de combustible en vez de unir nombres alternativos.

# PIP

In [ ]:
pip install matplotlib

In [ ]:
pip install kagglehub

In [1]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.0 MB/s  0:00:00


# LIBRERIAS

In [2]:
try:
  import numpy as np
  import matplotlib.pyplot as plt
  import pandas as pd
  import seaborn as sns
  import re # RegEx
  import matplotlib.pyplot as plt
  from sklearn.preprocessing import StandardScaler
  from sklearn.cluster import KMeans
  import gc

  #========= KAGGLE =========#
  import kagglehub

  #======= Rapidfuzz ========#
  from rapidfuzz.distance import Levenshtein

  print('Librerias importadas!')
except Exception as e:
  print(f'Librerias no importadas:\n{e}')

Librerias importadas!


# PREPROCESAMIENTO

In [3]:
# PETICION DE DATA HACIA KAGGLE

path = kagglehub.dataset_download("danielabosch/fuel-transactions-from-gas-pump")
print('DATASET DE KAGGLE CARGADO!')
print(path)

100%|██████████| 21.3M/21.3M [00:00<00:00, 129MB/s]

Extracting files...


DATASET DE KAGGLE CARGADO!
/root/.cache/kagglehub/datasets/danielabosch/fuel-transactions-from-gas-pump/versions/4


In [4]:
data_path = path+'/transacciones_surtidor.csv'

In [5]:
df = pd.read_csv(data_path, sep=',')
df.head()

/tmp/ipykernel_368/1672446801.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, sep=',')


,unique_id_transa,id_transaccion,id_vehiculo,id_usuario,id_equipo,veh_site_id,pump_site_id,tank_site_id,user_site_id,producto,...,temp_final,volumen_comp_15_inicial,volumen_comp_15_final,cantidad_comp_15,codigo_producto,geo_latitud,geo_longitud,id_empresa,id_industria,industria
0,377220101009999,9999,90952,12708,377,000001,1,1,0000,0.0,...,NaN,NaN,NaN,0.000,NaN,NaN,NaN,238,1.0,Transportation
1,504220101006768,6768,101866,6683,504,000001,1,1,0000,0.0,...,NaN,NaN,NaN,158.416,Product 1,NaN,NaN,25,5.0,Distributor
2,534220101007334,7334,123844,34280,534,009593,1,1,3532,0.0,...,30.2,17836.50,17728.5,107.850,0000,NaN,NaN,273,1.0,Transportation
3,878220101002206,2206,141769,46520,878,000103,2,1,0000,0.0,...,NaN,6921.42,NaN,80.002,GOM G3,NaN,NaN,1142,2.0,Mining
4,898220101005224,5224,129356,29688,898,999014,2,1,0001,0.0,...,NaN,NaN,NaN,734.900,Diesel 500,NaN,NaN,1083,1.0,Transportation


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 561747 entries, 0 to 561746
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   unique_id_transa         561747 non-null  int64  
 1   id_transaccion           561747 non-null  int64  
 2   id_vehiculo              561747 non-null  int64  
 3   id_usuario               561747 non-null  int64  
 4   id_equipo                561747 non-null  int64  
 5   veh_site_id              561740 non-null  object 
 6   pump_site_id             561747 non-null  int64  
 7   tank_site_id             561747 non-null  int64  
 8   user_site_id             545281 non-null  object 
 9   producto                 561644 non-null  float64
 10  id_bomba                 561747 non-null  int64  
 11  id_tanque                561747 non-null  int64  
 12  departamento             561747 non-null  object 
 13  cantidad                 561747 non-null  float64
 14  codi

In [ ]:
print(f'Cantidad de filas:\n{df.shape[0]}')
print(f'Cantidad de columnas:\n{df.shape[1]}')

Cantidad de filas:
561747
Cantidad de columnas:
32


In [ ]:
# DEFINICION DE COLUMNAS NUMERICAS Y CATEGORICAS
col_categoricas = df.select_dtypes(include=['object']).columns.tolist()
col_numericas = df.select_dtypes(exclude=['object','bool']).columns.tolist()
col_bools = df.select_dtypes(include=['bool']).columns.tolist()

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).sort_values("pct", ascending=False)

nulos

,nulos,pct
geo_latitud,555355,98.862121
geo_longitud,555355,98.862121
volumen_final,217475,38.714047
temp_final,217475,38.714047
volumen_comp_15_final,217475,38.714047
codigo_producto,175530,31.247163
temp_inicial,147072,26.181181
volumen_inicial,147072,26.181181
volumen_comp_15_inicial,147072,26.181181
industria,43118,7.675697


In [ ]:
df.columns

Index(['unique_id_transa', 'id_transaccion', 'id_vehiculo', 'id_usuario',
       'id_equipo', 'veh_site_id', 'pump_site_id', 'tank_site_id',
       'user_site_id', 'producto', 'id_bomba', 'id_tanque', 'departamento',
       'cantidad', 'codigo_error', 'timestamp', 'timestamp_stop',
       'nombre_prod', 'veh_efficiency', 'volumen_inicial', 'volumen_final',
       'temp_inicial', 'temp_final', 'volumen_comp_15_inicial',
       'volumen_comp_15_final', 'cantidad_comp_15', 'codigo_producto',
       'geo_latitud', 'geo_longitud', 'id_empresa', 'id_industria',
       'industria'],
      dtype='object')

In [ ]:
df.head(1)

,unique_id_transa,id_transaccion,id_vehiculo,id_usuario,id_equipo,veh_site_id,pump_site_id,tank_site_id,user_site_id,producto,...,temp_final,volumen_comp_15_inicial,volumen_comp_15_final,cantidad_comp_15,codigo_producto,geo_latitud,geo_longitud,id_empresa,id_industria,industria
0,377220101009999,9999,90952,12708,377,000001,1,1,0000,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,238,1.0,Transportation


In [6]:
df = df[[
    'unique_id_transa',
    'producto',
    'id_bomba',
    'cantidad',
    'cantidad_comp_15',
    'codigo_error',
    'timestamp',
    'timestamp_stop',
    'nombre_prod',
    'id_industria',
    'industria'
]]

In [7]:
df.head(2)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,Producto 0,1.0,Transportation
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,Product 1,5.0,Distributor


In [8]:
df.loc[df.duplicated()].shape[0]

0

In [8]:
df['industria'] = df['industria'].fillna('Unknown')
df['nombre_prod'] = df['nombre_prod'].fillna('Unknown')

In [10]:
df[['id_industria','industria']].loc[(df['id_industria'] == 0) & (df['industria']!='Unknown')].shape[0]

0

In [9]:
df['id_industria'] = df['id_industria'].fillna(0)
df['id_industria'].unique()

array([ 1.,  5.,  2.,  4.,  6.,  3.,  0.,  9.,  7.,  8., 10.])

In [10]:
df['industria'].unique()

array(['Transportation', 'Distributor', 'Mining', 'Agriculture',
       'Service Station', 'Oil&Gas', 'Unknown', 'Industry',
       'Construction', 'Public', 'Telcos'], dtype=object)

In [13]:
df.loc[pd.to_datetime(df['timestamp'], errors='coerce').isna()].shape[0], df.loc[pd.to_datetime(df['timestamp_stop'], errors='coerce').isna()].shape[0]

(0, 0)

In [11]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['timestamp_stop'] = pd.to_datetime(df['timestamp_stop'], errors='coerce')

In [12]:
df.head(2)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,Producto 0,1.0,Transportation
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,Product 1,5.0,Distributor


# CLASIFICACION POR TIPO COMBUSTIBLE

In [13]:
df.loc[df['nombre_prod'].isna()].shape[0]

0

In [14]:
len(df['nombre_prod'].unique())

102

In [15]:
df['nombre_prod'].unique()

array(['Producto 0', 'Product 1', 'Diesel 500', 'Gasoil GOM G3',
       'Ultra Diesel', 'VP Nafta', 'Super', 'Gasoil GOM G3 Diesel',
       'Infinia Diesel', 'Nafta Super', 'V-Power Nafta', 'Evolux Diesel',
       'V-Power Diesel', 'Diesel', 'Product 3', 'Gasoil', 'VP Diesel',
       'Diesel Grado 2', 'GASOIL', 'Gas Oil Grado 2', 'Gas Oil',
       'Gasoil Grado 3', 'EURO', 'COMUN', 'GOM', 'GASOIL 2', 'Product 2',
       'Producto 1', 'GAS-OIL - GRADO 3', 'nafta', 'DIESEL', 'VP-Diesel',
       'Formula', 'Urea', 'Infinia', 'Producto 2', 'GAS OIL ',
       'Gasoil GOM', 'INFINIA DIESEL', 'GAS OIL', 'Gasoil Comun',
       'Euro Diesel', 'DIESEL-500', 'Ultra Diesel ', 'DIESEL 1',
       'COMBUSTIBLE -  DIESEL 500', 'Gasoil Infinia', 'Diesel Premium',
       'Diesel Comun ', 'Axion Diesel', 'ULTRADIESEL', 'EURODIESEL',
       'Euro-Diesel', 'Ultra-Diesel', 'Nafta', 'Gasoil 500',
       'Diesel 500 PPM', 'S1002050010', 'Puerto Seco La Puerta SA',
       'Cereales Norte SA', 'Gas Oil Comun Gr

In [16]:
df['nombre_prod'] = df['nombre_prod'].str.rstrip().str.lstrip().str.upper()
df['nombre_prod'] = df['nombre_prod'].replace(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', '', regex=True)

In [23]:
df['tipo_producto'] = None
df.head(2)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria,tipo_producto
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,PRODUCTO 0,1.0,Transportation,None
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,PRODUCT 1,5.0,Distributor,None


In [93]:
tipo_diesel = ['infinia','diesel','diésel','euro',
               'combustible','gasoil','gas oil','gom',
               'gas-oil','ultra','formula', 'ud']

tipo_nafta = ['nafta','super']

tipo_urea = ['urea','azul']

# Join une cada valor de las listas con el operando '|' (or)
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_diesel), case=False), 'tipo_producto'] = 'Diesel'
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_nafta), case=False), 'tipo_producto'] = 'Nafta'
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_urea), case=False), 'tipo_producto'] = 'Urea'
df['tipo_producto'] = df['tipo_producto'].fillna('Otro')

In [31]:
df.head(3)

,unique_id_transa,producto,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,id_industria,industria,tipo_producto
0,377220101009999,0.0,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,PRODUCTO 0,1.0,Transportation,Otro
1,504220101006768,0.0,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,PRODUCT 1,5.0,Distributor,Otro
2,534220101007334,0.0,10922,109.515,107.850,C1,2022-01-01 02:21:27,2022-01-01 02:23:55,DIESEL 500,1.0,Transportation,Diesel


In [33]:
df['nombre_prod'].unique().shape[0]

80

In [40]:
similares = pd.DataFrame(columns=['Producto','Similar','Similitud'])

similares['Producto'] = [p for p in df['nombre_prod'].unique() if p]
lista_similar = []
lista_scores = []

for p in similares['Producto']:
  if isinstance(p, str):
    producto = p

    result = [(Levenshtein.normalized_similarity(producto, str(p_similar)), str(p_similar))
              for p_similar in similares['Producto']
              if (Levenshtein.normalized_similarity(producto, str(p_similar)) != 1)]

    score, similar = max(result, key=lambda x : x[0])

    lista_scores.append(score)
    lista_similar.append(similar)

similares['Similar'] = lista_similar
similares['Similitud'] = lista_scores

In [43]:
dict(zip(similares.Producto, similares.Similar))

{'PRODUCTO 0': 'PRODUCTO 1',
 'PRODUCT 1': 'PRODUCTO 1',
 'DIESEL 500': 'DIESEL500',
 'GASOIL GOM G3': 'GASOIL GOM',
 'ULTRA DIESEL': 'ULTRADIESEL',
 'VP NAFTA': 'VPOWER NAFTA',
 'SUPER': 'NAFTA SUPER',
 'GASOIL GOM G3 DIESEL': 'GASOIL GOM G3',
 'INFINIA DIESEL': 'INFINIA DIÉSEL',
 'NAFTA SUPER': 'SUPER',
 'VPOWER NAFTA': 'VP NAFTA',
 'EVOLUX DIESEL': 'VPOWER DIESEL',
 'VPOWER DIESEL': 'VP DIESEL',
 'DIESEL': 'VPDIESEL',
 'PRODUCT 3': 'PRODUCT 1',
 'GASOIL': 'GAS OIL',
 'VP DIESEL': 'VPDIESEL',
 'DIESEL GRADO 2': 'DIESEL GRADO 3',
 'GAS OIL GRADO 2': 'GASOIL GRADO 2',
 'GAS OIL': 'GASOIL',
 'GASOIL GRADO 3': 'GASOIL  GRADO 3',
 'EURO': 'EURO G3',
 'COMUN': 'FORMULA',
 'GOM': 'COMUN',
 'GASOIL 2': 'GASOIL 1',
 'PRODUCT 2': 'PRODUCTO 2',
 'PRODUCTO 1': 'PRODUCTO 0',
 'GASOIL  GRADO 3': 'GASOIL GRADO 3',
 'NAFTA': 'VP NAFTA',
 'VPDIESEL': 'VP DIESEL',
 'FORMULA': 'FORMULA DIESEL',
 'UREA': 'SUPER',
 'INFINIA': 'INFINIA T40',
 'PRODUCTO 2': 'PRODUCTO 0',
 'GASOIL GOM': 'GASOIL G3',
 'GASOI

In [77]:
#ARREGLO MANUAL A PARTIR DE SIMILARES
dict_similares = {'PRODUCTO 2  DIESEL' : 'PRODUCTO 2',
 'PRODUCT 1': 'PRODUCTO 1',
 'VP NAFTA': 'VPOWER NAFTA',
 'SUPER': 'NAFTA SUPER',
 'GASOIL GOM G3 DIESEL': 'GASOIL GOM G3',
 'VP DIESEL': 'VPOWER DIESEL',
 'VPDIESEL': 'VPOWER DIESEL',
 'GAS OIL GRADO 2': 'GASOIL GRADO 2',
 'GAS OIL': 'GASOIL',
 'GASOIL GOM': 'GOM',
 'GASOIL  GRADO 3': 'GASOIL GRADO 3',
 'PRODUCT 2': 'PRODUCTO 2',
 'FORMULA': 'FORMULA DIESEL',
 'EURO DIESEL': 'EURODIESEL',
 'DIESEL500': 'DIESEL 500',
 'ULTRADIESEL': 'ULTRA DIESEL',
 'GAS OIL 500': 'GASOIL 500',
 'INFINIA DIÉSEL': 'INFINIA DIESEL',
 'PRODUCTO 1  EURO DIESEL': 'PRODUCTO 1 EURO DIESEL',
 'D 500': 'DIESEL 500',
 'GASOIL MARÍTIMO': 'GOM',
 'EURO': 'EURODIESEL',
 'DIESEL EURO': 'EURODIESEL'}

In [87]:
combustibles = pd.DataFrame(columns=['Producto'])
combustibles['Producto'] = [p for p in df['nombre_prod'].unique() if p]
combustibles

,Producto
0,PRODUCTO 0
1,PRODUCT 1
2,DIESEL 500
3,GASOIL GOM G3
4,ULTRA DIESEL
...,...
75,GOM DIESEL 500
76,DIESEL 500 G2
77,COMBUSTIBLE
78,QUANTIUM DIESEL


In [89]:
combustibles['Producto'] = combustibles.replace({'Producto' : dict_similares})

In [90]:
combustibles['Producto'].unique()

array(['PRODUCTO 0', 'PRODUCTO 1', 'DIESEL 500', 'GASOIL GOM G3',
       'ULTRA DIESEL', 'VPOWER NAFTA', 'NAFTA SUPER', 'INFINIA DIESEL',
       'EVOLUX DIESEL', 'VPOWER DIESEL', 'DIESEL', 'PRODUCT 3', 'GASOIL',
       'DIESEL GRADO 2', 'GASOIL GRADO 2', 'GASOIL GRADO 3', 'EURODIESEL',
       'COMUN', 'GOM', 'GASOIL 2', 'PRODUCTO 2', 'NAFTA',
       'FORMULA DIESEL', 'UREA', 'INFINIA', 'GASOIL COMUN', 'DIESEL 1',
       'COMBUSTIBLE   DIESEL 500', 'GASOIL INFINIA', 'DIESEL PREMIUM',
       'DIESEL COMUN', 'AXION DIESEL', 'GASOIL 500', 'DIESEL 500 PPM',
       'S1002050010', 'PUERTO SECO LA PUERTA SA', 'CEREALES NORTE SA',
       'GAS OIL COMUN GR 2', 'GAS OIL EURO GR 3', 'DIESEL GRADO 3',
       'PRODUCTO 1 EURO DIESEL', 'EURO DIESEL GRADO3', 'GASOIL G3',
       'GASOIL 1', 'INFINIA T40', 'GLIFOSATO', 'AZUL 32', 'B100',
       'BIODIESEL', 'INFINIA T20', 'UD', 'UNKNOWN', 'GOM  DIESEL 500',
       'DIESEL 500 G2', 'COMBUSTIBLE', 'QUANTIUM DIESEL', 'EURO G3'],
      dtype=object)

In [81]:
combustibles['Producto'].unique().shape[0]

57

In [91]:
df['nombre_prod'] = df['nombre_prod'].map(dict_similares).fillna(df['nombre_prod'])

In [95]:
df.columns

Index(['unique_id_transa', 'producto', 'id_bomba', 'cantidad',
       'cantidad_comp_15', 'codigo_error', 'timestamp', 'timestamp_stop',
       'nombre_prod', 'id_industria', 'industria', 'tipo_producto'],
      dtype='object')

In [107]:
df = df[['unique_id_transa', 'id_bomba', 'cantidad',
       'cantidad_comp_15', 'codigo_error', 'timestamp', 'timestamp_stop',
       'nombre_prod', 'tipo_producto', 'id_industria', 'industria']]

In [103]:
df.head()

,unique_id_transa,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,tipo_producto,id_industria,industria
0,377220101009999,11435,0.000,0.000,UE,2022-01-01 03:49:21,2022-01-01 03:49:21,PRODUCTO 0,Otro,1.0,Transportation
1,504220101006768,10980,158.416,158.416,C4,2022-01-01 00:00:36,2022-01-01 00:02:57,PRODUCTO 1,Otro,5.0,Distributor
2,534220101007334,10922,109.515,107.850,C1,2022-01-01 02:21:27,2022-01-01 02:23:55,DIESEL 500,Diesel,1.0,Transportation
3,878220101002206,102713,80.002,80.002,C0,2022-01-01 02:42:11,2022-01-01 02:42:11,GASOIL GOM G3,Diesel,2.0,Mining
4,898220101005224,102408,734.900,734.900,BF,2022-01-01 00:15:56,2022-01-01 00:27:25,DIESEL 500,Diesel,1.0,Transportation


In [100]:
df['codigo_error'] = df['codigo_error'].str.upper()
df['codigo_error'].unique()

array(['UE', 'C4', 'C1', 'C0', 'BF', 'ES', 'C2', 'BE', 'BD', 'UL', 'BA',
       'RS', 'H2', 'H1', 'H4', 'RC', 'H5', 'ER', 'H3', 'E1', 'UD', 'PR',
       'B0', 'U3', '00', 'BR', 'EE', 'U4'], dtype=object)

In [108]:
df['duracion_carga_seg'] = (df['timestamp_stop'] - df['timestamp']).dt.total_seconds()

#Lunes=0, Domingo=6
df['dia_de_semana'] = df['timestamp'].dt.dayofweek

In [110]:
df['duracion_carga_seg'].head(3)

,duracion_carga_seg
0,0.0
1,141.0
2,148.0


In [127]:
df.loc[df['duracion_carga_seg'] > 30*60]

,unique_id_transa,id_bomba,cantidad,cantidad_comp_15,codigo_error,timestamp,timestamp_stop,nombre_prod,tipo_producto,id_industria,industria,duracion_carga_seg,dia_de_semana
686,569220101009816,10951,990.002,991.19,BF,2022-01-01 15:23:11,2022-01-01 16:23:50,GASOIL,Diesel,2.0,Mining,3639.0,5
749,898220101005228,102410,2500.000,2500.00,C0,2022-01-01 14:54:04,2022-01-01 15:49:34,DIESEL 500,Diesel,1.0,Transportation,3330.0,5
1315,851220101000836,102515,2819.480,2790.44,BF,2022-01-01 22:30:00,2022-01-01 23:02:33,DIESEL 500,Diesel,1.0,Transportation,1953.0,5
1316,851220102000837,102515,3039.430,3007.82,BF,2022-01-02 00:46:51,2022-01-02 01:21:56,DIESEL 500,Diesel,1.0,Transportation,2105.0,6
1328,898220101005232,102408,2500.000,2500.00,C0,2022-01-01 22:24:32,2022-01-01 22:57:56,DIESEL 500,Diesel,1.0,Transportation,2004.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
561321,1118220410000223,102998,8206.550,8236.92,BF,2022-04-10 08:22:35,2022-04-10 12:39:17,GOM DIESEL 500,Diesel,0.0,Unknown,15402.0,6
561337,1118220412000239,102998,6997.020,7036.90,BF,2022-04-12 08:29:38,2022-04-12 12:21:29,GOM DIESEL 500,Diesel,0.0,Unknown,13911.0,1
561348,1121220404000005,103006,0.000,0.00,BD,2022-04-04 21:39:59,2022-04-05 05:58:05,PRODUCTO 0,Otro,5.0,Distributor,29886.0,0
561377,1146220410000388,103041,3430.000,3408.05,C1,2022-04-10 00:01:17,2022-04-10 00:53:31,PRODUCTO 0,Otro,4.0,Agriculture,3134.0,6


In [120]:
round(df['duracion_carga_seg'].describe())

,duracion_carga_seg
count,561747.0
mean,256.0
std,4553.0
min,0.0
25%,0.0
50%,92.0
75%,182.0
max,1225117.0


# CODIGO VIEJO

In [ ]:
productos = [p.rstrip().lstrip().upper() for p in df['nombre_prod'].unique() if p]
productos = [re.sub(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', '', p) for p in productos if p]

productos = list(set(productos))

combustibles = pd.DataFrame(columns=['Producto','Tipo'])
combustibles['Producto'] = productos
combustibles.head(3)

,Producto,Tipo
0,EURO DIESEL,NaN
1,AZUL 32,NaN
2,COMBUSTIBLE DIESEL 500,NaN


In [ ]:
combustibles.shape[0]

80

In [ ]:
# TIPOS DE PRODUCTO:

tipo_diesel = ['infinia','diesel','diésel','euro',
               'combustible','gasoil','gas oil','gom',
               'gas-oil','ultra','formula', 'ud']

tipo_nafta = ['nafta','super']

tipo_urea = ['urea','azul']

for p in combustibles['Producto']:
  if isinstance(p, str):
    producto = p.lower()

    if any(t in producto for t in tipo_diesel):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Diesel'

    if any(t in producto for t in tipo_nafta):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Nafta'

    if any(t in producto for t in tipo_urea):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Urea'

    combustibles['Tipo'] = combustibles['Tipo'].fillna('Otro')



In [ ]:
combustibles.loc[combustibles['Tipo'] == 'Diesel']

,Producto,Tipo
0,EURO DIESEL,Diesel
2,COMBUSTIBLE DIESEL 500,Diesel
3,GAS OIL COMUN GR 2,Diesel
5,DIESEL PREMIUM,Diesel
6,GASOIL,Diesel
8,GASOIL GOM,Diesel
9,BIODIESEL,Diesel
10,VPDIESEL,Diesel
11,DIESEL 500 G2,Diesel
12,INFINIA T20,Diesel


In [ ]:
combustibles['Tipo'].unique()

array(['Diesel', 'Urea', 'Otro', 'Nafta'], dtype=object)